In [1]:
pip install backtesting

Note: you may need to restart the kernel to use updated packages.


In [2]:
from backtesting import Backtest, Strategy
from backtesting.lib import crossover
import pandas as pd
import yfinance as yf

raw = yf.download("AAPL", start="2023-01-01", end="2024-12-31")
raw.columns = raw.columns.droplevel(1)    # flatten multi-index
raw = raw[["Open", "High", "Low", "Close", "Volume"]]
raw.dropna(inplace=True)   # removes any rows that have missing values
raw.head()                 # just displays the first 5 rows in the dataframe as a sanity check

C:\Users\purpl\anaconda3\Lib\site-packages\backtesting\_plotting.py:55: UserWarning: Jupyter Notebook detected. Setting Bokeh output to notebook. This may not work in Jupyter clients without JavaScript support, such as old IDEs. Reset with `backtesting.set_bokeh_output(notebook=False)`.
  warnings.warn('Jupyter Notebook detected. '


Loading BokehJS ...

[*********************100%***********************]  1 of 1 completed


Price,Open,High,Low,Close,Volume
Date,,,,,
2023-01-03,128.105761,128.715409,122.097730,122.982712,112117500
2023-01-04,124.772336,126.512801,122.992546,124.251183,89113600
2023-01-05,125.008335,125.637653,122.677892,122.933548,80962700
2023-01-06,123.907018,128.115581,122.805707,127.456764,87754700
2023-01-09,128.292595,131.183532,127.722273,127.977928,70790800


In [5]:
# --- Imports & Data ---
from backtesting import Backtest, Strategy
from backtesting.lib import crossover
import pandas as pd
import yfinance as yf

raw = yf.download("AAPL", start="2023-01-01", end="2024-12-31")
raw.columns = raw.columns.droplevel(1)    # flatten multi-index
raw = raw[["Open", "High", "Low", "Close", "Volume"]]
raw.dropna(inplace=True)   # removes any rows that have missing values

# --- Helper Functions ---
def compute_rsi(series, period=14):
    delta = series.diff()   #series = [[10, 14], [14, 16], [16, 12]]  delta = [+4, +2, -4]
    gain = delta.clip(lower=0)   #[4, 2, 0]
    loss = -delta.clip(upper=0)  #[0, 0, 4]
    avg_gain = gain.rolling(period).mean()
    avg_loss = loss.rolling(period).mean()
    rs = avg_gain / avg_loss
    return 100 - (100 / (1 + rs))

def compute_bb_lower(series, period=20, num_std=2):
    ma = series.rolling(period).mean()
    std = series.rolling(period).std()
    return ma - (num_std * std)

class RsiMaStrategy(Strategy):
    rsi_entry = 45
    rsi_exit = 70
    stop_loss_pct = 6
    
    def init(self):
        close = pd.Series(self.data.Close)
        self.rsi = self.I(compute_rsi, close)
        self.ma20 = self.I(lambda x: x.rolling(20).mean(), close)
        self.ma50 = self.I(lambda x: x.rolling(50).mean(), close)
        self.bb_lower = self.I(compute_bb_lower, close)

    def next(self):
        if len(self.data) < 51:    # wait until MA50 has enough data points
            return
        price = self.data.Close[-1]
        stop_price = price * (1 - self.stop_loss_pct / 100)

        if not self.position:
            if (self.rsi[-1] < self.rsi_entry 
                and self.ma20[-1] > self.ma50[-1]
                and price < self.bb_lower[-1]):
                self.buy(sl=stop_price)
        else:
            if self.rsi[-1] > self.rsi_exit:
                    self.position.close()
                
def optimize_ticker(ticker, start="2020-01-01", end="2024-12-31"):
    raw = yf.download(ticker, start=start, end=end, auto_adjust=True)
    raw.columns = raw.columns.droplevel(1)

    bt = Backtest(raw, RsiMaStrategy, cash=10000, commission=0.002)

    stats, heatmap = bt.optimize(
        rsi_entry=range(25,55,5),
        rsi_exit=range(60, 80, 5),
        stop_loss_pct=range(4, 10, 2),
        maximize='Return [%]',
        return_heatmap=True
    )

    return {
        'ticker': ticker,
        'rsi_entry': stats._strategy.rsi_entry,
        'rsi_exit': stats._strategy.rsi_exit,
        'stop_loss_pct': stats._strategy.stop_loss_pct,
        'return': stats['Return [%]'],
        'sharpe': stats['Sharpe Ratio'],
        'max_dd': stats['Max. Drawdown [%]']
    }

[*********************100%***********************]  1 of 1 completed


In [6]:
tickers = ['AAPL', 'MSFT', 'GOOGL', 'JPM', 'XOM']

results = []
for ticker in tickers:
    print(f"Optimizing {ticker}...")
    result = optimize_ticker(ticker)
    results.append(result)

results_df = pd.DataFrame(results)
print(results_df)

[*********************100%***********************]  1 of 1 completed

Optimizing AAPL...



C:\Users\purpl\anaconda3\Lib\site-packages\backtesting\backtesting.py:1624: RuntimeWarning: If you want to use multi-process optimization with `multiprocessing.get_start_method() == 'spawn'` (e.g. on Windows),set `backtesting.Pool = multiprocessing.Pool` (or of the desired context) and hide `bt.optimize()` call behind a `if __name__ == '__main__'` guard. Currently using thread-based paralellism, which might be slightly slower for non-numpy / non-GIL-releasing code. See https://github.com/kernc/backtesting.py/issues/1256
  output = _optimize_grid()


Backtest.optimize:   0%|          | 0/72 [00:00<?, ?it/s]

Backtest.run:   0%|          | 0/1207 [00:00<?, ?bar/s]

...

Backtest.run:   0%|          | 0/1207 [00:00<?, ?bar/s]

Backtest.run:   0%|          | 0/1207 [00:00<?, ?bar/s]

Backtest.run:   0%|          | 0/1207 [00:00<?, ?bar/s]

Backtest.run:   0%|          | 0/1207 [00:00<?, ?bar/s]

  ticker  rsi_entry  rsi_exit  stop_loss_pct     return    sharpe     max_dd
0   AAPL         30        65              8  34.124719  0.576610 -17.864254
1   MSFT         45        75              4  20.146053  0.357735 -24.214181
2  GOOGL         45        70              8  37.686827  0.506326 -31.220427
3    JPM         40        75              6  73.319212  1.005428 -15.826353
4    XOM         35        75              8  84.767597  0.719614 -15.555947


In [7]:
consensus = {
    'rsi_entry': int(results_df['rsi_entry'].median()),
    'rsi_exit': int(results_df['rsi_exit'].median()),
    'stop_loss_pct': int(results_df['stop_loss_pct'].median())
}

print(consensus)

{'rsi_entry': 40, 'rsi_exit': 75, 'stop_loss_pct': 8}


In [8]:
def run_with_params(ticker, rsi_entry, rsi_exit, stop_loss_pct, start="2020-01-01", end="2024-12-31"):
    raw = yf.download(ticker, start=start, end=end, auto_adjust=True)
    raw.columns = raw.columns.droplevel(1)
    bt = Backtest(raw, RsiMaStrategy, cash=10000, commission=0.002)

    stats = bt.run(
        rsi_entry=rsi_entry,
        rsi_exit=rsi_exit,
        stop_loss_pct=stop_loss_pct
    )

    return {
        'ticker': ticker,
        'return': stats['Return [%]'],
        'sharpe': stats['Sharpe Ratio'],
        'max_dd': stats['Max. Drawdown [%]']
    }

consensus_results = []
for ticker in tickers:
    print(f"Running {ticker}...")
    result = run_with_params(ticker, **consensus)
    consensus_results.append(result)

consensus_df = pd.DataFrame(consensus_results)
print(consensus_df)

[*********************100%***********************]  1 of 1 completed

Running AAPL...


Backtest.run:   0%|          | 0/1207 [00:00<?, ?bar/s]

[*********************100%***********************]  1 of 1 completed

Running MSFT...


...

Backtest.run:   0%|          | 0/1207 [00:00<?, ?bar/s]

C:\Users\purpl\AppData\Local\Temp\ipykernel_11860\451503177.py:6: UserWarning: Some trades remain open at the end of backtest. Use `Backtest(..., finalize_trades=True)` to close them and include them in stats.
  stats = bt.run(
[*********************100%***********************]  1 of 1 completed

Running XOM...


Backtest.run:   0%|          | 0/1207 [00:00<?, ?bar/s]

  ticker     return    sharpe     max_dd
0   AAPL  -7.547602 -0.121732 -40.772020
1   MSFT   0.675682  0.011855 -29.240763
2  GOOGL  11.168164  0.156778 -34.484533
3    JPM  50.732492  0.838478 -25.342782
4    XOM  58.897364  0.562993 -21.376459


In [7]:
for ticker in ['MSFT', 'GOOGL', 'JPM', 'XOM']:
    data = yf.download(ticker, start='2023-01-01', end='2024-12-31')
    data.columns = data.columns.droplevel(1)
    data = data[["Open", "High", "Low", "Close", "Volume"]]
    data.dropna(inplace=True)
    
    bt_test = Backtest(data, RsiMaStrategy, cash=10000, commission=0.002)
    s = bt_test.run()
    # print(stats[['# Trades', 'Win Rate [%]', 'Return [%]', 'Buy & Hold Return [%]', 'Sharpe Ratio', 'Max. Drawdown [%]']])
    print(f"{ticker}: Trades={s['# Trades']}, WinRate={s['Win Rate [%]']:.0f}%, Return={s['Return [%]']:.1f}%, Sharpe={s['Sharpe Ratio']:.2f}")

[*********************100%***********************]  1 of 1 completed


Backtest.run:   0%|          | 0/451 [00:00<?, ?bar/s]

[*********************100%***********************]  1 of 1 completed

MSFT: Trades=4, WinRate=50%, Return=-8.1%, Sharpe=-0.62


Backtest.run:   0%|          | 0/451 [00:00<?, ?bar/s]

...

C:\Users\purpl\AppData\Local\Temp\ipykernel_3660\4152937625.py:8: UserWarning: Some trades remain open at the end of backtest. Use `Backtest(..., finalize_trades=True)` to close them and include them in stats.
  s = bt_test.run()


JPM: Trades=5, WinRate=80%, Return=31.7%, Sharpe=1.55


[*********************100%***********************]  1 of 1 completed


Backtest.run:   0%|          | 0/451 [00:00<?, ?bar/s]

XOM: Trades=6, WinRate=33%, Return=-9.7%, Sharpe=-0.55


In [19]:
###### from backtesting import Backtest
import yfinance as yf

data = yf.download('AAPL', start='2023-01-01', end='2024-12-31')
data.columns = data.columns.droplevel(1)
# print(data.columns)
# print(data.head())

bt = Backtest(data, RsiMaStrategy, cash=10000, commission=0.002)

stats, heatmap = bt.optimize(
    rsi_entry=range(25, 55, 5),
    rsi_exit=range(60, 80, 5),
    stop_loss_pct=range(4, 10, 2),
    maximize='Return [%]',
    return_heatmap=True
)

print(stats._strategy)

[*********************100%***********************]  1 of 1 completed
C:\Users\purpl\anaconda3\Lib\site-packages\backtesting\backtesting.py:1624: RuntimeWarning: If you want to use multi-process optimization with `multiprocessing.get_start_method() == 'spawn'` (e.g. on Windows),set `backtesting.Pool = multiprocessing.Pool` (or of the desired context) and hide `bt.optimize()` call behind a `if __name__ == '__main__'` guard. Currently using thread-based paralellism, which might be slightly slower for non-numpy / non-GIL-releasing code. See https://github.com/kernc/backtesting.py/issues/1256
  output = _optimize_grid()


Backtest.optimize:   0%|          | 0/72 [00:00<?, ?it/s]

Backtest.run:   0%|          | 0/451 [00:00<?, ?bar/s]

Backtest.run:   0%|          | 0/451 [00:00<?, ?bar/s]

Backtest.run:   0%|          | 0/451 [00:00<?, ?bar/s]

...

Backtest.run:   0%|          | 0/451 [00:00<?, ?bar/s]

Backtest.run:   0%|          | 0/451 [00:00<?, ?bar/s]

Backtest.run:   0%|          | 0/451 [00:00<?, ?bar/s]

Backtest.run:   0%|          | 0/451 [00:00<?, ?bar/s]

RsiMaStrategy(rsi_entry=25,rsi_exit=65,stop_loss_pct=4)


In [20]:
data2 = yf.download('MSFT', start='2023-01-01', end='2024-12-31')
data2.columns = data2.columns.droplevel(1)

bt2 = Backtest(data2, RsiMaStrategy, cash=10000, commission=0.002)
stats2 = bt2.run(rsi_entry=25, rsi_exit=65, stop_loss_pct=4)
print(stats2[['# Trades', 'Win Rate [%]', 'Return [%]', 'Sharpe Ratio', 'Max. Drawdown [%]']])

[*********************100%***********************]  1 of 1 completed


Backtest.run:   0%|          | 0/451 [00:00<?, ?bar/s]

# Trades                    1
Win Rate [%]              0.0
Return [%]          -7.240941
Sharpe Ratio         -0.95652
Max. Drawdown [%]   -8.897096
dtype: object


In [7]:
bt.plot()

GridPlot(id='p1814', ...)